# FI_1000 based alarm filtering

In [ ]:
# Alarms based on 02FI_1000 levels
import pandas as pd
from IPython.display import display

LOWER_LIMIT_FI1000 = 8.386505779
UPPER_LIMIT_FI1000 = 8.630339925

alarm_workbook_path = '/home/h604827/ControlActions/RESULTS/03TIC_1023_episodes/03TIC_1023_pvlo_alarms_clustered_with_control_actions.xlsx'
alarm_sheet_name = 'alarm_clusters'
fallback_pv_data_path = '/home/h604827/ControlActions/DATA/03TIC_1023_JAN_2026.parquet'
fi1000_pv_data_path = '/home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026_filtered.parquet'
fi1000_col = '02FI_1000.PV'
expansion_schedule = [0.0, 0.05, 0.10, 0.15]
evaluation_lead_minutes = 60
P

def strip_timezone(datetime_series):
    if getattr(datetime_series.dt, 'tz', None) is not None:
        return datetime_series.dt.tz_localize(None)
    return datetime_series


def get_expanded_fi1000_limits(expansion_fraction):
    lower_limit = LOWER_LIMIT_FI1000 * (1 - expansion_fraction)
    upper_limit = UPPER_LIMIT_FI1000 * (1 + expansion_fraction)
    return lower_limit, upper_limit


if 'minute_pv_data' not in globals():
    minute_pv_data = pd.read_parquet(globals().get('pv_data_path', fallback_pv_data_path))

# Load FI1000 from separate parquet that contains it
fi1000_pv_data = pd.read_parquet(fi1000_pv_data_path, columns=[fi1000_col] if fi1000_col in pd.read_parquet(fi1000_pv_data_path, columns=[]).columns else None)
if fi1000_col not in fi1000_pv_data.columns:
    raise KeyError(f'{fi1000_col} was not found in {fi1000_pv_data_path}.')

if 'TimeStamp' in fi1000_pv_data.columns:
    fi1000_series = fi1000_pv_data[['TimeStamp', fi1000_col]].copy()
    fi1000_series['TimeStamp'] = pd.to_datetime(fi1000_series['TimeStamp'], errors='coerce')
    fi1000_series = fi1000_series.dropna(subset=['TimeStamp'])
    fi1000_series['TimeStamp'] = strip_timezone(fi1000_series['TimeStamp'])
    fi1000_series = fi1000_series.set_index('TimeStamp')[fi1000_col]
else:
    fi1000_series = fi1000_pv_data[[fi1000_col]].copy().reset_index()
    fi1000_series = fi1000_series.rename(columns={fi1000_series.columns[0]: 'TimeStamp'})
    fi1000_series['TimeStamp'] = pd.to_datetime(fi1000_series['TimeStamp'], errors='coerce')
    fi1000_series = fi1000_series.dropna(subset=['TimeStamp'])
    fi1000_series['TimeStamp'] = strip_timezone(fi1000_series['TimeStamp'])
    fi1000_series = fi1000_series.set_index('TimeStamp')[fi1000_col]

fi1000_series = pd.to_numeric(fi1000_series, errors='coerce').sort_index()
fi1000_series = fi1000_series[~fi1000_series.index.duplicated(keep='last')]

alarm_clusters_raw = pd.read_excel(alarm_workbook_path, sheet_name=alarm_sheet_name).copy()
raw_alarm_count = len(alarm_clusters_raw)

# Use cluster_start_time and cluster_end_time as alarm boundaries, deduplicate to unique clusters
for col in ['cluster_start_time', 'cluster_end_time']:
    alarm_clusters_raw[col] = pd.to_datetime(alarm_clusters_raw[col], errors='coerce')
    alarm_clusters_raw[col] = strip_timezone(alarm_clusters_raw[col])

alarm_windows = (
    alarm_clusters_raw[['cluster_start_time', 'cluster_end_time']]
    .drop_duplicates()
    .dropna()
    .rename(columns={'cluster_start_time': 'alarm_start', 'cluster_end_time': 'alarm_end'})
    .copy()
)
alarm_windows = alarm_windows.loc[alarm_windows['alarm_end'] >= alarm_windows['alarm_start']].copy()
alarm_windows = alarm_windows.sort_values('alarm_start').reset_index(drop=True)
alarm_windows['alarm_id'] = alarm_windows.index + 1
alarm_windows['evaluation_start'] = alarm_windows['alarm_start'] - pd.Timedelta(minutes=evaluation_lead_minutes)
alarm_windows['sample_start'] = alarm_windows['evaluation_start'].dt.ceil('min')
alarm_windows['sample_end'] = alarm_windows['alarm_end'].dt.floor('min')


def evaluate_alarm_window(sample_start, sample_end, lower_limit, upper_limit):
    if pd.isna(sample_start) or pd.isna(sample_end) or sample_end < sample_start:
        return pd.Series(
            {
                'expected_samples': 0,
                'observed_samples': 0,
                'full_coverage': False,
                'outside_limit_count': 0,
                'min_fi1000': float('nan'),
                'max_fi1000': float('nan'),
                'within_limits': False,
            }
        )

    window = fi1000_series.loc[sample_start:sample_end]
    expected_samples = int((sample_end - sample_start) / pd.Timedelta(minutes=1)) + 1
    observed_samples = len(window)
    valid_window = window.dropna()
    full_coverage = observed_samples == expected_samples and len(valid_window) == expected_samples

    if valid_window.empty:
        min_value = float('nan')
        max_value = float('nan')
        outside_limit_count = 0
    else:
        min_value = valid_window.min()
        max_value = valid_window.max()
        outside_limit_count = int(
            (~valid_window.between(lower_limit, upper_limit, inclusive='both')).sum()
        )

    within_limits = bool(full_coverage and outside_limit_count == 0)

    return pd.Series(
        {
            'expected_samples': expected_samples,
            'observed_samples': observed_samples,
            'full_coverage': full_coverage,
            'outside_limit_count': outside_limit_count,
            'min_fi1000': min_value,
            'max_fi1000': max_value,
            'within_limits': within_limits,
        }
    )


summary_rows = []
fi1000_alarm_pass_tables = {}

print(
    'Envelope expansion rule: lower the lower limit by the stated percentage of the lower limit and raise the upper limit by the stated percentage of the upper limit.'
 )
print(
    f'Evaluation window per alarm: [alarm_start - {evaluation_lead_minutes} minutes, alarm_end].'
 )
print(f'Total rows in alarm_clusters sheet: {raw_alarm_count}')
print(f'Unique alarm clusters (by cluster_start_time, cluster_end_time): {len(alarm_windows)}')

for expansion_fraction in expansion_schedule:
    expansion_pct = int(expansion_fraction * 100)
    lower_limit, upper_limit = get_expanded_fi1000_limits(expansion_fraction)

    scenario_eval = alarm_windows.apply(
        lambda row: evaluate_alarm_window(
            row['sample_start'],
            row['sample_end'],
            lower_limit,
            upper_limit,
        ),
        axis=1,
    )

    scenario_results = pd.concat(
        [
            alarm_windows[
                ['alarm_id', 'alarm_start', 'alarm_end', 'evaluation_start', 'sample_start', 'sample_end']
            ],
            scenario_eval,
        ],
        axis=1,
    )
    
    passing_alarms = scenario_results.loc[scenario_results['within_limits']].copy()
    passing_alarms['min_fi1000'] = passing_alarms['min_fi1000'].round(6)
    passing_alarms['max_fi1000'] = passing_alarms['max_fi1000'].round(6)
    fi1000_alarm_pass_tables[expansion_pct] = passing_alarms[
        [
            'alarm_id',
            'alarm_start',
            'alarm_end',
            'evaluation_start',
            'min_fi1000',
            'max_fi1000',
            'expected_samples',
        ]
    ].reset_index(drop=True)

    passing_alarm_count = len(passing_alarms)
    total_alarm_count = len(scenario_results)
    full_coverage_alarm_count = int(scenario_results['full_coverage'].sum())

    summary_rows.append(
        {
            'scenario': 'base_limits' if expansion_pct == 0 else f'base_plus_{expansion_pct}pct',
            'expansion_pct': expansion_pct,
            'lower_limit': round(lower_limit, 6),
            'upper_limit': round(upper_limit, 6),
            'passing_alarm_count': passing_alarm_count,
            'total_alarm_count': total_alarm_count,
            'passing_alarm_pct': round(100 * passing_alarm_count / total_alarm_count, 2),
            'full_coverage_alarm_count': full_coverage_alarm_count,
            'not_full_coverage_alarm_count': total_alarm_count - full_coverage_alarm_count,
        }
    )

fi1000_alarm_envelope_summary = pd.DataFrame(summary_rows)
display(fi1000_alarm_envelope_summary)

for expansion_pct in [0, 5, 10, 15]:
    passing_alarms = fi1000_alarm_pass_tables[expansion_pct]
    scenario_label = 'Base limits' if expansion_pct == 0 else f'Base limits + {expansion_pct}% limit-based expansion'
    print(
        f'\n{scenario_label}: {len(passing_alarms)} alarms stay within the evaluated FI_1000 limits for the full evaluation window.'
    )
    display(passing_alarms)

Envelope expansion rule: lower the lower limit by the stated percentage of the lower limit and raise the upper limit by the stated percentage of the upper limit.
Evaluation window per alarm: [alarm_start - 60 minutes, alarm_end].
Total rows in alarm_clusters sheet: 861
Unique alarm clusters (by cluster_start_time, cluster_end_time): 765


,scenario,expansion_pct,lower_limit,upper_limit,passing_alarm_count,total_alarm_count,passing_alarm_pct,full_coverage_alarm_count,not_full_coverage_alarm_count
0,base_limits,0,8.386506,8.630340,9,765,1.18,726,39
1,base_plus_5pct,5,7.967180,9.061857,374,765,48.89,726,39
2,base_plus_10pct,10,7.547855,9.493374,476,765,62.22,726,39
3,base_plus_15pct,15,7.128530,9.924891,574,765,75.03,726,39



Base limits: 9 alarms stay within the evaluated FI_1000 limits for the full evaluation window.


,alarm_id,alarm_start,alarm_end,evaluation_start,min_fi1000,max_fi1000,expected_samples
0,309,2023-02-09 03:53:24.554,2023-02-09 04:05:27.555,2023-02-09 02:53:24.554,8.393218,8.543584,72
1,316,2023-02-15 06:00:29.903,2023-02-15 06:05:11.904,2023-02-15 05:00:29.903,8.447051,8.568615,65
2,403,2023-03-14 20:29:07.252,2023-03-14 20:43:34.502,2023-03-14 19:29:07.252,8.446601,8.625230,74
3,410,2023-03-16 21:04:51.005,2023-03-16 21:13:50.103,2023-03-16 20:04:51.005,8.398504,8.596842,69
4,575,2024-09-24 05:53:39.105,2024-09-24 07:25:46.002,2024-09-24 04:53:39.105,8.391728,8.607897,152
5,576,2024-09-25 05:59:08.804,2024-09-25 06:15:35.305,2024-09-25 04:59:08.804,8.466553,8.628772,76
6,594,2024-10-08 07:22:17.255,2024-10-08 07:52:06.303,2024-10-08 06:22:17.255,8.397607,8.626418,90
7,625,2024-10-16 02:45:08.909,2024-10-16 02:56:31.653,2024-10-16 01:45:08.909,8.410912,8.610074,71
8,676,2024-11-04 08:21:59.154,2024-11-04 08:28:39.203,2024-11-04 07:21:59.154,8.423147,8.617540,67



Base limits + 5% limit-based expansion: 374 alarms stay within the evaluated FI_1000 limits for the full evaluation window.


,alarm_id,alarm_start,alarm_end,evaluation_start,min_fi1000,max_fi1000,expected_samples
0,50,2022-04-19 21:55:21.452,2022-04-19 22:07:35.455,2022-04-19 20:55:21.452,8.314299,8.441252,72
1,51,2022-04-20 00:21:31.252,2022-04-20 00:30:54.751,2022-04-19 23:21:31.252,8.288568,8.428849,69
2,53,2022-04-21 20:44:11.603,2022-04-21 21:26:30.602,2022-04-21 19:44:11.603,8.100166,8.321611,102
3,64,2022-04-25 20:21:38.354,2022-04-25 20:32:45.353,2022-04-25 19:21:38.354,8.156195,8.341634,71
4,67,2022-04-27 20:07:16.902,2022-04-27 21:07:10.403,2022-04-27 19:07:16.902,8.014865,8.229545,120
...,...,...,...,...,...,...,...
369,748,2024-11-27 20:29:45.305,2024-11-27 20:33:12.304,2024-11-27 19:29:45.305,8.341139,8.623547,64
370,749,2024-11-28 00:47:17.002,2024-11-28 00:57:37.004,2024-11-27 23:47:17.002,8.326220,8.665396,70
371,750,2024-11-28 06:16:09.603,2024-11-28 07:38:21.853,2024-11-28 05:16:09.603,8.213426,8.656194,142
372,763,2024-12-01 17:38:31.206,2024-12-01 18:28:21.256,2024-12-01 16:38:31.206,8.098107,8.469946,110



Base limits + 10% limit-based expansion: 476 alarms stay within the evaluated FI_1000 limits for the full evaluation window.


,alarm_id,alarm_start,alarm_end,evaluation_start,min_fi1000,max_fi1000,expected_samples
0,29,2022-03-22 19:49:10.203,2022-03-22 20:11:01.452,2022-03-22 18:49:10.203,7.834907,8.210807,82
1,31,2022-03-25 20:11:27.353,2022-03-25 20:19:38.602,2022-03-25 19:11:27.353,7.746611,7.926193,68
2,50,2022-04-19 21:55:21.452,2022-04-19 22:07:35.455,2022-04-19 20:55:21.452,8.314299,8.441252,72
3,51,2022-04-20 00:21:31.252,2022-04-20 00:30:54.751,2022-04-19 23:21:31.252,8.288568,8.428849,69
4,52,2022-04-20 20:31:42.902,2022-04-20 20:38:27.903,2022-04-20 19:31:42.902,7.668100,8.116641,67
...,...,...,...,...,...,...,...
471,749,2024-11-28 00:47:17.002,2024-11-28 00:57:37.004,2024-11-27 23:47:17.002,8.326220,8.665396,70
472,750,2024-11-28 06:16:09.603,2024-11-28 07:38:21.853,2024-11-28 05:16:09.603,8.213426,8.656194,142
473,757,2024-11-29 15:17:08.854,2024-11-29 15:27:41.602,2024-11-29 14:17:08.854,7.765778,8.146162,70
474,763,2024-12-01 17:38:31.206,2024-12-01 18:28:21.256,2024-12-01 16:38:31.206,8.098107,8.469946,110



Base limits + 15% limit-based expansion: 574 alarms stay within the evaluated FI_1000 limits for the full evaluation window.


,alarm_id,alarm_start,alarm_end,evaluation_start,min_fi1000,max_fi1000,expected_samples
0,26,2022-03-19 03:49:41.753,2022-03-19 03:57:22.252,2022-03-19 02:49:41.753,7.189519,7.491353,68
1,27,2022-03-19 04:36:19.503,2022-03-19 04:59:59.002,2022-03-19 03:36:19.503,7.200479,7.491353,83
2,28,2022-03-20 02:29:15.153,2022-03-20 02:38:34.153,2022-03-20 01:29:15.153,7.362760,7.552870,69
3,29,2022-03-22 19:49:10.203,2022-03-22 20:11:01.452,2022-03-22 18:49:10.203,7.834907,8.210807,82
4,31,2022-03-25 20:11:27.353,2022-03-25 20:19:38.602,2022-03-25 19:11:27.353,7.746611,7.926193,68
...,...,...,...,...,...,...,...
569,759,2024-11-30 02:47:55.054,2024-11-30 02:58:40.802,2024-11-30 01:47:55.054,7.293752,7.839723,71
570,761,2024-11-30 15:37:18.804,2024-11-30 17:28:15.604,2024-11-30 14:37:18.804,7.142010,7.969434,171
571,762,2024-12-01 02:28:45.110,2024-12-01 02:36:08.103,2024-12-01 01:28:45.110,7.286981,7.911726,68
572,763,2024-12-01 17:38:31.206,2024-12-01 18:28:21.256,2024-12-01 16:38:31.206,8.098107,8.469946,110


,TimeStamp,03FIC_3435.PV,03FI_1151.PV,03HIC_1023A.OP,03HIC_1023B.OP,03LIC_1016.OP,03LIC_1016.PV,03LIC_1031.OP,03LIC_1031.PV,03LIC_1034.OP,...,03TI_1015.PV,03TI_1024.PV,03TI_1108.PV,03TI_1404.PV,03TI_1405.PV,03TI_3121.PV,03TI_3123.PV,03TI_3410.PV,AlarmStatus,AlarmType
1737580,2025-06-23 20:40:00,99517.820,269300.47,24.409676,99.527740,41.182150,37.799980,0.985968,45.035300,34.861706,...,17.378136,17.965654,46.974907,39.214390,35.882633,-6.031647,-5.744507,-5.294479,OFF,
1737581,2025-06-23 20:41:00,99575.984,274164.06,26.332031,100.000000,40.494507,40.594604,0.884887,45.451090,35.158040,...,17.643166,18.239292,46.884323,39.299576,35.920494,-6.054237,-5.716282,-5.351208,OFF,
1737582,2025-06-23 20:42:00,99098.984,272590.56,23.468270,98.774610,37.065247,43.634050,1.680819,47.313790,35.285645,...,17.477524,18.154371,46.839035,39.365833,36.166590,-6.026009,-5.800976,-5.341755,OFF,
1737583,2025-06-23 20:43:00,98749.980,271760.88,24.142128,99.313705,40.652360,38.010810,1.242190,45.464920,35.569447,...,17.361572,18.003399,46.816390,39.621387,36.355896,-6.054237,-5.730400,-5.351208,OFF,
1737584,2025-06-23 20:44:00,99064.090,276367.00,27.067520,100.000000,41.669228,38.224674,0.815796,45.180603,35.849243,...,17.643166,18.258163,46.714478,39.649784,36.336964,-6.054237,-5.688046,-5.417393,OFF,


In [16]:
# Understand FI1000 distribution during PIC_1104 alarm episodes
# For each alarm window, compute min/max/mean of FI1000

fi1000_stats_per_alarm = []
for _, row in alarm_windows.iterrows():
    sample_start = row['sample_start']
    sample_end = row['sample_end']
    if pd.isna(sample_start) or pd.isna(sample_end):
        continue
    window = fi1000_series.loc[sample_start:sample_end].dropna()
    if window.empty:
        continue
    fi1000_stats_per_alarm.append({
        'alarm_id': row['alarm_id'],
        'alarm_start': row['alarm_start'],
        'alarm_end': row['alarm_end'],
        'fi1000_min': window.min(),
        'fi1000_max': window.max(),
        'fi1000_mean': window.mean(),
        'fi1000_median': window.median(),
        'fi1000_range': window.max() - window.min(),
        'n_samples': len(window),
    })

fi1000_stats_df = pd.DataFrame(fi1000_stats_per_alarm)

print(f"FI1000 stats across {len(fi1000_stats_df)} PIC_1104 alarm episodes (evaluation window: alarm_start - {evaluation_lead_minutes}min to alarm_end):")
print(f"\nOverall FI1000 distribution during alarms:")
print(f"  Min of mins:    {fi1000_stats_df['fi1000_min'].min():.4f}")
print(f"  Max of maxs:    {fi1000_stats_df['fi1000_max'].max():.4f}")
print(f"  Mean of means:  {fi1000_stats_df['fi1000_mean'].mean():.4f}")
print(f"  Median of medians: {fi1000_stats_df['fi1000_median'].median():.4f}")
print(f"\nPer-episode FI1000 range (max - min within each episode):")
print(fi1000_stats_df['fi1000_range'].describe().to_string())

print(f"\nFI1000 mean per episode distribution:")
print(fi1000_stats_df['fi1000_mean'].describe().to_string())

print(f"\nCurrent limits being tested: [{LOWER_LIMIT_FI1000:.6f}, {UPPER_LIMIT_FI1000:.6f}]")
print(f"15% expanded: [{LOWER_LIMIT_FI1000 * 0.85:.6f}, {UPPER_LIMIT_FI1000 * 1.15:.6f}]")

# Show how many episodes have their MEAN within each range
for lo, hi in [(7.0, 7.5), (7.5, 8.0), (8.0, 8.25), (8.25, 8.5), (8.5, 8.75), (8.75, 9.0), (9.0, 9.5), (9.5, 10.0), (10.0, 11.0)]:
    count = ((fi1000_stats_df['fi1000_mean'] >= lo) & (fi1000_stats_df['fi1000_mean'] <= hi)).sum()
    if count > 0:
        print(f"  Mean FI1000 in [{lo:.2f}, {hi:.2f}]: {count} episodes")

# Show how many episodes have ALL values within each range (strict)
print(f"\nStrict (all values within range):")
for lo, hi in [(7.0, 7.5), (7.5, 8.0), (8.0, 8.5), (8.5, 9.0), (9.0, 9.5), (9.5, 10.0), (10.0, 11.0)]:
    count = ((fi1000_stats_df['fi1000_min'] >= lo) & (fi1000_stats_df['fi1000_max'] <= hi)).sum()
    if count > 0:
        print(f"  All FI1000 in [{lo:.2f}, {hi:.2f}]: {count} episodes")

display(fi1000_stats_df.sort_values('fi1000_mean').head(20))

FI1000 stats across 125 PIC_1104 alarm episodes (evaluation window: alarm_start - 60min to alarm_end):

Overall FI1000 distribution during alarms:
  Min of mins:    1.8592
  Max of maxs:    9.0673
  Mean of means:  6.2730
  Median of medians: 6.2580

Per-episode FI1000 range (max - min within each episode):
count    125.000000
mean       0.853343
std        0.840410
min        0.200340
25%        0.426859
50%        0.570726
75%        0.816984
max        6.037673

FI1000 mean per episode distribution:
count    125.000000
mean       6.272982
std        0.629525
min        4.365783
25%        5.911997
50%        6.278954
75%        6.519935
max        8.355572

Current limits being tested: [8.386506, 8.630340]
15% expanded: [7.128530, 9.924891]
  Mean FI1000 in [7.00, 7.50]: 9 episodes
  Mean FI1000 in [7.50, 8.00]: 2 episodes
  Mean FI1000 in [8.00, 8.25]: 1 episodes
  Mean FI1000 in [8.25, 8.50]: 1 episodes

Strict (all values within range):
  All FI1000 in [7.00, 7.50]: 1 episodes


,alarm_id,alarm_start,alarm_end,fi1000_min,fi1000_max,fi1000_mean,fi1000_median,fi1000_range,n_samples
72,79,2022-04-10 10:37:37.903,2022-04-10 14:13:49.902,4.022843,4.726278,4.365783,4.353866,0.703435,276
73,80,2022-04-11 22:33:31.603,2022-04-12 06:54:21.853,4.425683,4.772205,4.583897,4.584387,0.346522,561
63,70,2022-03-03 22:35:58.504,2022-03-03 22:41:22.504,4.430171,5.112055,4.892540,4.914823,0.681884,56
74,81,2022-04-12 18:38:19.703,2022-04-12 19:43:50.804,4.600978,6.362349,5.261640,5.273941,1.761371,125
44,51,2022-02-09 18:59:11.103,2022-02-09 19:15:55.104,4.829263,6.228643,5.325896,5.199292,1.399380,75
104,113,2023-07-30 04:54:09.802,2023-07-30 05:05:01.802,5.031032,5.707711,5.335833,5.310355,0.676679,71
103,112,2023-07-30 00:07:23.903,2023-07-30 00:16:19.904,5.004008,5.673614,5.342179,5.314013,0.669606,69
105,114,2023-07-30 05:54:20.902,2023-07-30 06:41:17.903,5.029368,5.667565,5.357462,5.361836,0.638197,107
106,115,2023-07-30 07:15:45.704,2023-07-30 08:33:40.253,5.020240,5.737572,5.371708,5.366958,0.717332,138
36,43,2022-02-04 05:33:59.803,2022-02-04 06:15:32.805,4.706291,6.471170,5.398338,4.842896,1.764879,102


In [17]:
fi1000_stats_df

,alarm_id,alarm_start,alarm_end,fi1000_min,fi1000_max,fi1000_mean,fi1000_median,fi1000_range,n_samples
0,7,2022-01-03 23:02:40.454,2022-01-04 00:18:21.102,6.502116,6.824732,6.643682,6.650488,0.322616,94
1,8,2022-01-04 04:00:32.703,2022-01-04 04:10:55.703,5.934086,6.415094,6.083353,6.076758,0.481008,70
2,9,2022-01-04 05:10:03.004,2022-01-04 06:08:02.102,6.049519,6.666984,6.421251,6.502738,0.617465,118
3,10,2022-01-04 20:21:46.652,2022-01-04 21:20:22.706,5.951471,6.446469,6.192131,6.155546,0.494998,119
4,11,2022-01-04 22:33:06.752,2022-01-04 22:50:54.003,6.059290,6.404289,6.213744,6.179791,0.344999,77
...,...,...,...,...,...,...,...,...,...
120,129,2024-01-28 08:11:28.708,2024-01-28 08:40:19.004,6.659685,7.020843,6.833174,6.841626,0.361158,89
121,130,2024-01-29 08:19:55.803,2024-01-29 08:30:18.303,6.482701,7.053428,6.732697,6.704482,0.570726,71
122,131,2024-05-01 23:53:19.303,2024-05-02 00:13:54.553,5.947065,8.601021,7.276926,7.006796,2.653956,80
123,132,2024-12-03 18:49:46.802,2024-12-03 18:54:49.803,5.718953,6.288197,5.961249,5.953914,0.569244,65


In [6]:
import shutil
from pathlib import Path

episodes = fi1000_alarm_pass_tables[5]['alarm_id'].tolist()

SOURCE = Path('/home/h604827/ControlActions/RESULTS/03TIC_1023_episodes/episode_visualizations')
DEST = Path('/home/h604827/ControlActions/RESULTS/1023_episodes_fi1000_filtered')

if DEST.exists():
    shutil.rmtree(DEST)
DEST.mkdir(parents=True, exist_ok=True)

copied = 0
missing = []
for ep in episodes:
    src = SOURCE / f'episode_{ep:04d}'
    dst = DEST / f'episode_{ep:04d}'
    if src.exists():
        shutil.copytree(src, dst)
        copied += 1
    else:
        missing.append(ep)

print(f'Copied {copied}/{len(episodes)} episodes to {DEST}')
if missing:
    print(f'Missing from source ({len(missing)}): {missing}')

Copied 374/374 episodes to /home/h604827/ControlActions/RESULTS/1023_episodes_fi1000_filtered


In [12]:
# Check how many of the 276 FI1000-filtered episodes have operator control actions (OP/SP)
filtered_episode_ids = fi1000_alarm_pass_tables[5]['alarm_id'].tolist()

# Get control actions and match to these episodes
ca = control_actions.copy()
for col in ['cluster_start', 'cluster_end', 'VT_Start']:
    ca[col] = pd.to_datetime(ca[col], errors='coerce')
    ca[col] = strip_timezone(ca[col])

# Build lookup for the 276 passing alarms
passing_lookup = fi1000_alarm_pass_tables[5][['alarm_id', 'alarm_start', 'alarm_end']].copy()
passing_lookup = passing_lookup.rename(columns={'alarm_start': 'cluster_start', 'alarm_end': 'cluster_end'})

# Match control actions to these episodes
matched_actions = ca.merge(passing_lookup, on=['cluster_start', 'cluster_end'], how='inner')

# Filter to OP/SP only
op_sp_actions = matched_actions[matched_actions['Description'].isin(['OP', 'SP'])]

episodes_with_actions = op_sp_actions['alarm_id'].nunique()
episodes_without_actions = len(filtered_episode_ids) - episodes_with_actions

print(f"Total FI1000-filtered episodes (5% expansion): {len(filtered_episode_ids)}")
print(f"Episodes WITH operator control actions (OP/SP): {episodes_with_actions}")
print(f"Episodes WITHOUT operator control actions: {episodes_without_actions}")
print(f"\nEpisode IDs with actions: {sorted(op_sp_actions['alarm_id'].unique().tolist())}")
print(f"\nEpisode IDs without actions: {sorted(set(filtered_episode_ids) - set(op_sp_actions['alarm_id'].unique()))}")

Total FI1000-filtered episodes (5% expansion): 276
Episodes WITH operator control actions (OP/SP): 124
Episodes WITHOUT operator control actions: 152

Episode IDs with actions: [116, 117, 118, 119, 120, 121, 123, 127, 128, 132, 133, 134, 147, 148, 149, 156, 167, 180, 181, 183, 184, 198, 199, 200, 201, 205, 206, 210, 216, 217, 230, 235, 236, 242, 243, 267, 268, 273, 277, 278, 295, 297, 298, 299, 304, 305, 306, 307, 310, 313, 318, 325, 327, 328, 329, 330, 332, 333, 334, 335, 337, 339, 360, 361, 362, 365, 366, 367, 368, 371, 383, 386, 408, 414, 415, 417, 419, 420, 421, 422, 427, 434, 446, 450, 452, 453, 461, 462, 465, 466, 467, 473, 474, 483, 484, 485, 491, 492, 494, 495, 496, 497, 498, 499, 500, 504, 508, 513, 524, 530, 533, 535, 539, 540, 548, 552, 560, 562, 563, 564, 566, 567, 577, 578]

Episode IDs without actions: [115, 129, 130, 131, 135, 136, 137, 138, 144, 145, 146, 150, 151, 152, 154, 155, 163, 164, 165, 166, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 182, 190, 1

In [14]:
df = pd.read_parquet('/home/h604827/ControlActions/DATA/25_tags_events_preprocessed/51fda50f-24a3-4632-9803-7ec34744c34b.parquet')
df['AreaName'].value_counts()

AreaName
1E    70213
Name: count, dtype: int64

In [ ]:
# output_path = "/home/h604827/ControlActions/329_alarm_timestamps.csv"
# fi1000_alarm_pass_tables[15].to_csv(output_path, index=False)
# print(f"Saved: {output_path}")

Saved: /home/h604827/ControlActions/329_alarm_timestamps.csv


In [4]:
# Most operated tags for alarms within the FI_1000 15% limit-based expansion band
import pandas as pd
from IPython.display import display

target_expansion_fraction = 0.15
target_expansion_pct = int(target_expansion_fraction * 100)
target_lower_limit, target_upper_limit = get_expanded_fi1000_limits(target_expansion_fraction)

fi1000_target_band_eval = alarm_windows.apply(
    lambda row: evaluate_alarm_window(
        row['sample_start'],
        row['sample_end'],
        target_lower_limit,
        target_upper_limit,
    ),
    axis=1,
 )

fi1000_target_band_results = pd.concat(
    [
        alarm_windows[
            ['alarm_id', 'alarm_start', 'alarm_end', 'evaluation_start', 'sample_start', 'sample_end']
        ],
        fi1000_target_band_eval,
    ],
    axis=1,
 )

fi1000_target_band_passing_alarms = fi1000_target_band_results.loc[
    fi1000_target_band_results['within_limits']
].copy()
fi1000_target_band_passing_alarms['min_fi1000'] = fi1000_target_band_passing_alarms['min_fi1000'].round(6)
fi1000_target_band_passing_alarms['max_fi1000'] = fi1000_target_band_passing_alarms['max_fi1000'].round(6)

if 'control_actions' not in globals():
    control_actions = pd.read_excel(
        alarm_workbook_path,
        sheet_name='control_actions',
    )

filtered_control_actions = control_actions.copy()
for col in ['cluster_start', 'cluster_end', 'VT_Start']:
    filtered_control_actions[col] = pd.to_datetime(filtered_control_actions[col], errors='coerce')
    filtered_control_actions[col] = strip_timezone(filtered_control_actions[col])

qualified_alarm_lookup = fi1000_target_band_passing_alarms[
    ['alarm_id', 'alarm_start', 'alarm_end']
].rename(columns={'alarm_start': 'cluster_start', 'alarm_end': 'cluster_end'})

qualified_alarm_actions = filtered_control_actions.merge(
    qualified_alarm_lookup,
    on=['cluster_start', 'cluster_end'],
    how='inner',
 )

qualified_alarm_actions = qualified_alarm_actions.loc[
    qualified_alarm_actions['Description'].isin(['OP', 'SP'])
].copy()

if qualified_alarm_actions.empty:
    raise ValueError(
        'No OP/SP control-action rows were matched to the alarms inside the FI_1000 target limit-based expansion band.'
    )

action_count_by_tag = (
    qualified_alarm_actions.groupby(['Description', 'Source'])
    .agg(
        action_count=('Source', 'size'),
        unique_alarm_count=('alarm_id', 'nunique'),
    )
    .reset_index()
    .rename(columns={'Description': 'action_type', 'Source': 'tag'})
 )

timing_breakdown = (
    qualified_alarm_actions.groupby(['Description', 'Source', 'action_timing'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={'Description': 'action_type', 'Source': 'tag'})
 )

tag_operation_summary = action_count_by_tag.merge(
    timing_breakdown,
    on=['action_type', 'tag'],
    how='left',
 )

sort_cols = ['action_count', 'unique_alarm_count', 'tag']
op_most_operated_tags = (
    tag_operation_summary.loc[tag_operation_summary['action_type'] == 'OP']
    .sort_values(sort_cols, ascending=[False, False, True])
    .reset_index(drop=True)
 )
sp_most_operated_tags = (
    tag_operation_summary.loc[tag_operation_summary['action_type'] == 'SP']
    .sort_values(sort_cols, ascending=[False, False, True])
    .reset_index(drop=True)
 )

print(
    f'FI_1000 {target_expansion_pct}% limit-based expansion band: [{target_lower_limit:.6f}, {target_upper_limit:.6f}]'
 )
print(
    f'Evaluation window per alarm: [alarm_start - {evaluation_lead_minutes} minutes, alarm_end].'
 )
print(
    f'{len(fi1000_target_band_passing_alarms)} alarms stay inside this band for the full evaluation window.'
 )
print(
    f"{qualified_alarm_actions['alarm_id'].nunique()} of those alarms have matched OP/SP control-action rows, totaling {len(qualified_alarm_actions)} action rows."
 )

display(
    fi1000_target_band_passing_alarms[
        ['alarm_id', 'alarm_start', 'alarm_end', 'evaluation_start', 'min_fi1000', 'max_fi1000', 'expected_samples']
    ].reset_index(drop=True)
 )

print('\nMost operated tags - OP actions')
display(op_most_operated_tags)

print('\nMost operated tags - SP actions')
display(sp_most_operated_tags)

FI_1000 15% limit-based expansion band: [7.128530, 9.924891]
Evaluation window per alarm: [alarm_start - 60 minutes, alarm_end].
329 alarms stay inside this band for the full evaluation window.
243 of those alarms have matched OP/SP control-action rows, totaling 5389 action rows.


,alarm_id,alarm_start,alarm_end,evaluation_start,min_fi1000,max_fi1000,expected_samples
0,50,2022-03-25 23:04:22.253,2022-03-25 23:10:18.501,2022-03-25 22:04:22.253,7.607256,7.887766,66
1,63,2022-04-13 09:47:48.906,2022-04-13 10:06:24.655,2022-04-13 08:47:48.906,7.212435,8.089724,79
2,64,2022-04-14 16:07:34.456,2022-04-14 16:58:01.954,2022-04-14 15:07:34.456,8.168119,8.475675,111
3,67,2022-05-10 20:49:35.355,2022-05-10 20:57:28.601,2022-05-10 19:49:35.355,7.368787,7.695838,68
4,79,2022-06-20 15:59:09.706,2022-06-20 16:03:21.452,2022-06-20 14:59:09.706,8.102098,8.342938,64
...,...,...,...,...,...,...,...
324,535,2025-06-21 22:15:17.359,2025-06-21 22:16:18.517,2025-06-21 21:15:17.359,8.535002,8.738719,61
325,536,2025-06-22 13:13:26.264,2025-06-22 13:50:05.278,2025-06-22 12:13:26.264,7.907859,8.260180,97
326,537,2025-06-22 14:32:47.231,2025-06-22 14:59:39.544,2025-06-22 13:32:47.231,7.779396,8.201122,87
327,538,2025-06-22 15:55:07.321,2025-06-22 16:04:21.651,2025-06-22 14:55:07.321,7.715930,7.871311,69



Most operated tags - OP actions


,action_type,tag,action_count,unique_alarm_count,after,before,during
0,OP,03PIC_1013,1419,95,227,603,589
1,OP,03FIC_3435,801,106,161,298,342
2,OP,03HIC_1151,601,60,100,224,277
3,OP,03LIC_1071,525,21,112,80,333
4,OP,03HIC_3132,141,12,23,101,17
5,OP,03HIC_3100,135,24,65,43,27
6,OP,03HIC_1141,128,16,44,53,31
7,OP,03FIC_3435A,116,1,0,0,116
8,OP,03FIC_3415,90,15,17,50,23
9,OP,03FIC_1085,77,4,32,27,18



Most operated tags - SP actions


,action_type,tag,action_count,unique_alarm_count,after,before,during
0,SP,03LIC_1034,445,66,139,161,145
1,SP,03LIC_1071,202,39,66,8,128
2,SP,03PIC_3131,114,3,0,6,108
3,SP,03TIC_1009,110,32,31,44,35
4,SP,03LIC_1016,75,14,10,4,61
5,SP,03LIC_1085,68,24,15,18,35
6,SP,03PIC_1068,14,4,4,9,1
7,SP,03LIC_3408,11,7,1,3,7
8,SP,03LIC_3153,10,4,3,6,1
9,SP,03LIC_1097,5,5,3,1,1


In [10]:
# Alarms within the FI_1000 target limit-based expansion band that have no matched OP/SP control actions
matched_op_sp_alarm_ids = set(qualified_alarm_actions['alarm_id'].unique())

fi1000_target_band_alarms_without_op_sp_actions = (
    fi1000_target_band_passing_alarms.loc[
        ~fi1000_target_band_passing_alarms['alarm_id'].isin(matched_op_sp_alarm_ids),
        [
            'alarm_id',
            'alarm_start',
            'alarm_end',
            'evaluation_start',
            'min_fi1000',
            'max_fi1000',
            'expected_samples',
        ],
    ]
    .sort_values('alarm_id')
    .reset_index(drop=True)
 )

print(
    f"{len(fi1000_target_band_alarms_without_op_sp_actions)} alarms inside the FI_1000 {target_expansion_pct}% limit-based expansion band have no matched OP/SP control-action rows."
 )
display(fi1000_target_band_alarms_without_op_sp_actions)

86 alarms inside the FI_1000 15% limit-based expansion band have no matched OP/SP control-action rows.


,alarm_id,alarm_start,alarm_end,evaluation_start,min_fi1000,max_fi1000,expected_samples
0,116,2022-12-08 07:59:18.355,2022-12-08 08:06:23.353,2022-12-08 06:59:18.355,8.169575,8.430376,67
1,140,2023-03-04 21:45:00.803,2023-03-04 21:48:48.804,2023-03-04 20:45:00.803,7.462617,7.773997,63
2,145,2023-03-05 06:33:19.455,2023-03-05 06:37:20.955,2023-03-05 05:33:19.455,7.427788,7.773242,64
3,167,2023-04-14 20:41:06.303,2023-04-14 21:09:26.303,2023-04-14 19:41:06.303,7.390894,7.896458,88
4,171,2023-05-02 04:42:34.805,2023-05-02 04:49:03.810,2023-05-02 03:42:34.805,8.239440,8.611371,67
...,...,...,...,...,...,...,...
81,487,2025-01-16 11:07:26.293,2025-01-16 11:08:34.304,2025-01-16 10:07:26.293,8.442484,8.759841,61
82,490,2025-04-19 17:22:41.581,2025-04-19 17:23:50.656,2025-04-19 16:22:41.581,8.477488,8.805232,61
83,491,2025-04-20 05:52:40.613,2025-04-20 05:53:46.723,2025-04-20 04:52:40.613,8.251720,8.859434,61
84,492,2025-04-20 07:29:40.425,2025-04-20 07:35:44.424,2025-04-20 06:29:40.425,8.236081,8.683028,66


In [11]:
# Alarms within the FI_1000 target limit-based expansion band that have matched OP/SP control actions
fi1000_target_band_alarms_with_op_sp_actions = (
    fi1000_target_band_passing_alarms.loc[
        fi1000_target_band_passing_alarms['alarm_id'].isin(matched_op_sp_alarm_ids),
        [
            'alarm_id',
            'alarm_start',
            'alarm_end',
            'evaluation_start',
            'min_fi1000',
            'max_fi1000',
            'expected_samples',
        ],
    ]
    .sort_values('alarm_id')
    .reset_index(drop=True)
 )

print(
    f"{len(fi1000_target_band_alarms_with_op_sp_actions)} alarms inside the FI_1000 {target_expansion_pct}% limit-based expansion band have matched OP/SP control-action rows."
 )
display(fi1000_target_band_alarms_with_op_sp_actions)

243 alarms inside the FI_1000 15% limit-based expansion band have matched OP/SP control-action rows.


,alarm_id,alarm_start,alarm_end,evaluation_start,min_fi1000,max_fi1000,expected_samples
0,50,2022-03-25 23:04:22.253,2022-03-25 23:10:18.501,2022-03-25 22:04:22.253,7.607256,7.887766,66
1,63,2022-04-13 09:47:48.906,2022-04-13 10:06:24.655,2022-04-13 08:47:48.906,7.212435,8.089724,79
2,64,2022-04-14 16:07:34.456,2022-04-14 16:58:01.954,2022-04-14 15:07:34.456,8.168119,8.475675,111
3,67,2022-05-10 20:49:35.355,2022-05-10 20:57:28.601,2022-05-10 19:49:35.355,7.368787,7.695838,68
4,79,2022-06-20 15:59:09.706,2022-06-20 16:03:21.452,2022-06-20 14:59:09.706,8.102098,8.342938,64
...,...,...,...,...,...,...,...
238,535,2025-06-21 22:15:17.359,2025-06-21 22:16:18.517,2025-06-21 21:15:17.359,8.535002,8.738719,61
239,536,2025-06-22 13:13:26.264,2025-06-22 13:50:05.278,2025-06-22 12:13:26.264,7.907859,8.260180,97
240,537,2025-06-22 14:32:47.231,2025-06-22 14:59:39.544,2025-06-22 13:32:47.231,7.779396,8.201122,87
241,538,2025-06-22 15:55:07.321,2025-06-22 16:04:21.651,2025-06-22 14:55:07.321,7.715930,7.871311,69


In [12]:
# Alarm IDs only: FI_1000 target limit-based expansion band alarms without matched OP/SP control actions
no_op_sp_alarm_ids = fi1000_target_band_alarms_without_op_sp_actions['alarm_id'].tolist()
print(no_op_sp_alarm_ids)

[116, 140, 145, 167, 171, 187, 189, 190, 192, 199, 200, 203, 223, 226, 227, 228, 229, 230, 232, 233, 234, 235, 236, 237, 238, 241, 242, 244, 245, 246, 247, 248, 249, 252, 290, 294, 295, 297, 298, 303, 306, 307, 308, 318, 319, 320, 321, 322, 325, 326, 329, 331, 338, 339, 341, 351, 353, 374, 382, 384, 394, 413, 415, 417, 419, 420, 422, 423, 424, 427, 430, 431, 435, 442, 450, 455, 464, 477, 478, 479, 482, 487, 490, 491, 492, 513]


In [5]:
# Episodes where 03LIC_1071 was operated (OP actions) — identify the 21 episodes
# and show the most operated tags across those episodes

# Find the 21 alarm_ids where 03LIC_1071 had OP actions
lc1071_op_actions = qualified_alarm_actions.loc[
    (qualified_alarm_actions['Source'] == '03LIC_1071') &
    (qualified_alarm_actions['Description'] == 'OP')
]
lc1071_operated_alarm_ids = set(lc1071_op_actions['alarm_id'].unique())

print(f"Episodes where 03LIC_1071 was operated (OP): {len(lc1071_operated_alarm_ids)}")

# Get ALL actions (all tags) in those 21 episodes
actions_in_1071_operated_episodes = qualified_alarm_actions.loc[
    qualified_alarm_actions['alarm_id'].isin(lc1071_operated_alarm_ids)
].copy()

print(f"Total OP/SP action rows in those {len(lc1071_operated_alarm_ids)} episodes: {len(actions_in_1071_operated_episodes)}")
print(f"Unique tags operated: {actions_in_1071_operated_episodes['Source'].nunique()}")

# Most operated tags (by action count and unique episode count) in these 21 episodes
most_operated_in_1071_eps = (
    actions_in_1071_operated_episodes.groupby(['Description', 'Source'])
    .agg(
        action_count=('Source', 'size'),
        unique_episode_count=('alarm_id', 'nunique'),
    )
    .reset_index()
    .rename(columns={'Description': 'action_type', 'Source': 'tag'})
)

# Split OP vs SP
op_in_1071_eps = (
    most_operated_in_1071_eps.loc[most_operated_in_1071_eps['action_type'] == 'OP']
    .sort_values(['action_count', 'unique_episode_count', 'tag'], ascending=[False, False, True])
    .reset_index(drop=True)
)
sp_in_1071_eps = (
    most_operated_in_1071_eps.loc[most_operated_in_1071_eps['action_type'] == 'SP']
    .sort_values(['action_count', 'unique_episode_count', 'tag'], ascending=[False, False, True])
    .reset_index(drop=True)
)

print(f"\nMost operated tags (OP) in the {len(lc1071_operated_alarm_ids)} episodes where 03LIC_1071 was operated:")
display(op_in_1071_eps)

print(f"\nMost operated tags (SP) in the {len(lc1071_operated_alarm_ids)} episodes where 03LIC_1071 was operated:")
display(sp_in_1071_eps)

Episodes where 03LIC_1071 was operated (OP): 21
Total OP/SP action rows in those 21 episodes: 803
Unique tags operated: 19

Most operated tags (OP) in the 21 episodes where 03LIC_1071 was operated:


,action_type,tag,action_count,unique_episode_count
0,OP,03LIC_1071,525,21
1,OP,03FIC_1085,35,2
2,OP,03HIC_1141,26,2
3,OP,03LIC_1094,20,1
4,OP,03HIC_1151,17,1
5,OP,03HIC_3100,16,3
6,OP,03PIC_1013,11,2
7,OP,03XAX_1001,10,1
8,OP,03XAX_1002,10,1
9,OP,03FIC_3435,6,2



Most operated tags (SP) in the 21 episodes where 03LIC_1071 was operated:


,action_type,tag,action_count,unique_episode_count
0,SP,03LIC_1071,66,12
1,SP,03LIC_1034,21,3
2,SP,03LIC_1085,10,3
3,SP,03TIC_1009,7,2
4,SP,03LIC_1016,4,2
5,SP,03LIC_1094,2,1


In [6]:
lc1071_operated_alarm_ids

{np.int64(250),
 np.int64(414),
 np.int64(429),
 np.int64(432),
 np.int64(433),
 np.int64(434),
 np.int64(441),
 np.int64(470),
 np.int64(471),
 np.int64(472),
 np.int64(473),
 np.int64(474),
 np.int64(475),
 np.int64(480),
 np.int64(481),
 np.int64(485),
 np.int64(486),
 np.int64(488),
 np.int64(498),
 np.int64(499),
 np.int64(534)}